### Import & Clean Data

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

warnings.filterwarnings("ignore", category=FutureWarning)

csv_path = Path.cwd().parent / "data" / "all_fights.csv"
data_frame = pd.read_csv(csv_path)

data_frame = pd.read_csv(csv_path)
data_frame = data_frame[~data_frame["blue_stance"].isin(["Switch ", "Open Stance", "Unknown"])]
data_frame = data_frame[~data_frame["red_stance"].isin(["Switch ", "Open Stance", "Unknown"])]

# Drop Catch Weight from weight_class (72 entries)
data_frame = data_frame[~data_frame["weight_class"].isin(["Catch Weight"])]

# Drop reach diff over 40 (1 entry)
data_frame = data_frame[(data_frame["reach_diff"] > -40) & (data_frame["reach_diff"] < 40)]

# Drop round_diff outlier (1 entry)
data_frame = data_frame[(data_frame["rounds_diff"] > -100) & (data_frame["rounds_diff"] < 100)]

# Convert red_winner from string to bool
data_frame["red_winner"] = (data_frame["red_winner"].astype(str).str.lower() == "t").astype(int)
# Convert title bout from string to bool'
data_frame["title_bout"] = (data_frame["title_bout"].astype(str).str.lower() == "t").astype(int)

data_frame["fight_date"] = pd.to_datetime(data_frame["fight_date"])
data_frame = data_frame.sort_values("fight_date", kind="stable").reset_index(drop=True)

### Feature Engineering

In [7]:
SEED = 42
DATE = "2023-06-01"
SELECTION_METRIC = "roc_auc"

CAT_FEATURES = [
    "gender", "weight_class", "red_stance", "blue_stance"
]


extra_features = CAT_FEATURES + [
    "red_age", "blue_age", "b_match_wc_rank", "r_match_wc_rank"
]

# Build feature dataframe with all extras + all diffs (including odds)
all_features_df = pd.concat([data_frame[extra_features], data_frame.filter(regex="_diff")], axis=1)

# Age gap is more/less impactful depending on where it sits. A 25 v 28 age gap is less significant 
# than it being 32 v 35 and raw age_diff doesn't see that.
all_features_df["avg_age"] = (all_features_df.red_age + all_features_df.blue_age) / 2
all_features_df["avg_age_c"] = all_features_df["avg_age"] - all_features_df["avg_age"].mean()  # center it

# Create a new column that captures the interaction: how much age_diff's
# effect should be amplified or dampened based on how old the pair is on average
all_features_df["age_diff_x_avg"] = all_features_df["age_diff"] * all_features_df["avg_age_c"]

# Drop raw ages since we already have the affect
all_features_df = all_features_df.drop(columns=["red_age", "blue_age"])

# Replace instances of rank 20 (unranked) to 16 so ranking distribution is uniform
new_worst_rank = 16
worst_rank = all_features_df[["b_match_wc_rank", "r_match_wc_rank"]].max().max()
all_features_df["b_match_wc_rank"] = all_features_df["b_match_wc_rank"].replace(to_replace=worst_rank, value=new_worst_rank)
all_features_df["r_match_wc_rank"] = all_features_df["r_match_wc_rank"].replace(to_replace=worst_rank ,value=new_worst_rank)

### Benchmark

In [ ]:
# Target
y = data_frame["red_winner"]

# Temporal Split
train_mask, test_mask = data_frame['fight_date'] <= DATE, data_frame['fight_date'] > DATE

data_frame_test = data_frame[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

cv = TimeSeriesSplit(n_splits=5)
scoring = {"roc_auc": "roc_auc", "accuracy": "accuracy", "neg_log_loss": "neg_log_loss"}

majority = max(y_test.mean(), 1 - y_test.mean())

# The lower number is the favorite (-250 beats +215, -150 beats -110).
# odds_diff = red_odds - blue_odds, so < 0 means red is favored.
# Whhen equal odds, default to red, the majority class.
red_fav = (data_frame_test["red_odds"] <= data_frame_test["blue_odds"]).astype(int)
n_ties = int((data_frame_test["red_odds"] == data_frame_test["blue_odds"]).sum())
favorite = accuracy_score(y_test, red_fav)

# AUC / log loss for the favorite benchmark: convert odds to implied
# probabilities and remove the bookmaker margin (normalise to sum 1).
def implied(o):
    o = o.astype(float)
    return np.where(o < 0, -o / (-o + 100), 100 / (o + 100))

p_red, p_blue = implied(data_frame_test["red_odds"]), implied(data_frame_test["blue_odds"])
p = p_red / (p_red + p_blue)

results = [
    {"model": "Majority class (always red)", "features": "-",
        "test_acc": majority, "test_auc": 0.5,
        "test_logloss": log_loss(y_test, np.full(len(y_test), y_test.mean()))},
    {"model": "Betting favorite", "features": "odds only",
        "test_acc": favorite, "test_auc": roc_auc_score(y_test, p),
        "test_logloss": log_loss(y_test, p)},
]
print(f"Coin Flip fights in test set (equal odds, assigned to red): {n_ties}")

# Split into with/without odds
feature_sets = {
    "with odds": list(all_features_df.columns),
    "no odds": [c for c in all_features_df.columns if c != "odds_diff"],
}

Pick'em fights in test set (equal odds, assigned to red): 14


### Modeling

In [ ]:
for featureset_name, columns, in feature_sets.items():
    # Collect every non-categorical feature (continuous) into an array
    num_features = [c for c in columns if c not in CAT_FEATURES]
    X_train = all_features_df.loc[train_mask, columns]
    X_test = all_features_df.loc[test_mask, columns]

    ## == Build Models ==

    # --- Preprocessors ---
    preprocessor = ColumnTransformer([
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATURES),
    ])

    preprocessor_scaled = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), num_features),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CAT_FEATURES)
    ])

    # --- Param grids ---
    param_grid_lr = {
        "clf__C": [0.01, 0.1, 1.0, 10.0],
        "clf__l1_ratio": [0, 1],
        "clf__solver": ["liblinear"]
    }

    param_grid_rf = {
        "clf__n_estimators": [100, 300],
        "clf__max_depth": [5, 10, 20, None],
        "clf__min_samples_leaf": [5, 20, 45],
        "clf__max_features": ["sqrt", "log2", 0.5]
    }

    param_grid_xgb = {
        "clf__n_estimators": [100, 300],
        "clf__max_depth": [2, 3, 5],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0]
    }

    param_grid_cb = {
        "clf__iterations": [200, 500],
        "clf__learning_rate": [0.03, 0.05, 0.1],
        "clf__depth": [4, 6],
        "clf__l2_leaf_reg": [3, 10]
    }

    all_models = {
        # --- Pipelines ---
        "Logistic Regression" : (
            Pipeline([
                ("preprocess", preprocessor_scaled),
                ("clf", LogisticRegression(max_iter=5000, random_state=SEED))
            ]),
            param_grid_lr
        ),

        "Random Forest" : (
            Pipeline([
                ("preprocess", preprocessor),
                ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1))
            ]),
            param_grid_rf
        ),

        "XGBClassifier" : (
            Pipeline([
                ("preprocess", preprocessor),
                ("clf", XGBClassifier(eval_metric="logloss", random_state=SEED, n_jobs=-1))
            ]),
            param_grid_xgb
        ),

        "CatBoost": (
            Pipeline([
                ("clf", CatBoostClassifier(random_seed=SEED,
                                           verbose=0,
                                           thread_count=-1,
                                           allow_writing_files=False))
            ]),
            param_grid_cb,
        )
    }

    for model, (pipe, param) in all_models.items():
        t0 = time.time()
        grid_search = GridSearchCV(pipe, param, cv=cv, scoring=scoring, 
                                   refit=SELECTION_METRIC, n_jobs=-1, verbose=1,
        )

        fit_params = {}
        if model == "CatBoost":
            fit_params = ({"clf__cat_features": CAT_FEATURES})
        
        grid_search.fit(X_train, y_train, **fit_params)
        elapsed = time.time() - t0

        cvr = pd.DataFrame(grid_search.cv_results_)
        trial_cols = ["params", "mean_test_roc_auc", "std_test_roc_auc",
                        "mean_test_accuracy", "mean_test_neg_log_loss"]
        print(f"\n {model} ({featureset_name}) : {len(cvr)} configs, "f"{elapsed:.1f}s ")
        
        print(cvr[trial_cols].sort_values("mean_test_roc_auc", ascending=False).to_string(index=False))

        # Test set is used once, only for the CV-selected configuration.
        best = grid_search.best_estimator_
        proba = best.predict_proba(X_test)[:, 1]
        results.append({
            "model": model, "features": featureset_name,
            "cv_auc": grid_search.best_score_,
            "cv_acc": cvr.loc[grid_search.best_index_, "mean_test_accuracy"],
            "test_acc": accuracy_score(y_test, (proba >= 0.5).astype(int)),
            "test_auc": roc_auc_score(y_test, proba),
            "test_logloss": log_loss(y_test, proba),
            "best_params": grid_search.best_params_,
            "fit_seconds": round(elapsed, 1),
        })

Fitting 5 folds for each of 8 candidates, totalling 40 fits


/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was depreca


 Logistic Regression (with odds) : 8 configs, 2.7s 
                                                            params  mean_test_roc_auc  std_test_roc_auc  mean_test_accuracy  mean_test_neg_log_loss
{'clf__C': 0.01, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}           0.703874          0.023344            0.655720               -0.630459
 {'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}           0.700685          0.020810            0.654237               -0.616470
{'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}           0.695815          0.022197            0.654449               -0.619123
 {'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}           0.693849          0.023045            0.653814               -0.621372
 {'clf__C': 1.0, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}           0.693588          0.023162            0.652966               -0.622141
 {'clf__C': 1.0, 'clf__penalty': 'l2', 'clf__solver': 'libl

/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propa


 Random Forrest (with odds) : 72 configs, 40.0s 
                                                                                                      params  mean_test_roc_auc  std_test_roc_auc  mean_test_accuracy  mean_test_neg_log_loss
      {'clf__max_depth': 5, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 45, 'clf__n_estimators': 100}           0.696946          0.022345            0.657415               -0.621165
      {'clf__max_depth': 5, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 45, 'clf__n_estimators': 300}           0.696329          0.022425            0.655297               -0.621274
     {'clf__max_depth': 10, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 45, 'clf__n_estimators': 300}           0.695748          0.022585            0.654661               -0.621012
     {'clf__max_depth': 20, 'clf__max_features': 0.5, 'clf__min_samples_leaf': 45, 'clf__n_estimators': 300}           0.695662          0.022823            0.654025               -0.621046


/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/kormakur-snorrason/HR/Year3/Fall/MachineLearning/MachineLearningRepo/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was depreca


 Logistic Regression (no odds) : 8 configs, 1.0s 
                                                            params  mean_test_roc_auc  std_test_roc_auc  mean_test_accuracy  mean_test_neg_log_loss
{'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}           0.622675          0.016011            0.606568               -0.656951
 {'clf__C': 0.1, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}           0.621235          0.017415            0.607203               -0.657376
 {'clf__C': 0.1, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}           0.619048          0.017373            0.608686               -0.659635
 {'clf__C': 1.0, 'clf__penalty': 'l1', 'clf__solver': 'liblinear'}           0.616988          0.018619            0.607203               -0.660656
 {'clf__C': 1.0, 'clf__penalty': 'l2', 'clf__solver': 'liblinear'}           0.614448          0.019871            0.604873               -0.663516
{'clf__C': 10.0, 'clf__penalty': 'l1', 'clf__solver': 'liblin

### Results

In [11]:
summary = pd.DataFrame(results)
pd.set_option("display.width", 200)
print("\n Summary (models selected by CV ROC-AUC, then scored once on test) ")
print(summary.drop(columns=["best_params"]).round(4).to_string(index=False))
summary.to_csv("results_summary.csv", index=False)


 Summary (models selected by CV ROC-AUC, then scored once on test) 
                      model  features  test_acc  test_auc  test_logloss  cv_auc  cv_acc  fit_seconds
Majority class (always red)         -    0.5558    0.5000        0.6869     NaN     NaN          NaN
           Betting favorite odds only    0.7043    0.7563        0.5809     NaN     NaN          NaN
        Logistic Regression with odds    0.7023    0.7597        0.5839  0.7039  0.6557          3.0
             Random Forrest with odds    0.6990    0.7528        0.5886  0.6969  0.6574         40.6
              XGBClassifier with odds    0.6964    0.7547        0.5869  0.6990  0.6597         23.1
                   CatBoost with odds    0.6964    0.7555        0.5850  0.6974  0.6561         43.5
        KNeghborsClassifier with odds    0.6667    0.7127        0.6175  0.6438  0.6248          2.6
        Logistic Regression with odds    0.7023    0.7597        0.5839  0.7039  0.6557          2.7
             Random Fo